# 03 · Rule-Based Detection

This notebook implements and evaluates rule-based AML detection strategies that flag potentially suspicious transactions using transparent heuristics.

## Goals
- Define interpretable risk rules
- Generate rule-trigger indicators
- Review precision/coverage trade-offs


### Code Breakdown

**What this cell does:** Imports required Python libraries and dependencies.

**Key command:** `import pandas as pd`


In [16]:
import pandas as pd

df = pd.read_csv('../data/processed/transactions_features.csv')

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,txn_count_sender,total_sent,avg_sent,txn_per_step,amount_deviation,balance_diff,is_high_risk_type
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,1,9839.64,9839.64,1,0.0,9839.64,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1,1864.28,1864.28,1,0.0,1864.28,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,1,181.00,181.00,1,0.0,181.00,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,1,181.00,181.00,1,0.0,181.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,1,11668.14,11668.14,1,0.0,11668.14,0


### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df.columns`


In [17]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud', 'txn_count_sender', 'total_sent', 'avg_sent',
       'txn_per_step', 'amount_deviation', 'balance_diff',
       'is_high_risk_type'],
      dtype='object')

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_high_value'] = (df['amount'] > 200000).astype(int)`


In [18]:
df['rule_high_value'] = (df['amount'] > 200000).astype(int)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_high_risk_type'] = df['is_high_risk_type']`


In [19]:
df['rule_high_risk_type'] = df['is_high_risk_type']

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_account_emptied'] = (`


In [20]:
df['rule_account_emptied'] = (
    (df['oldbalanceOrg'] > 0) & 
    (df['newbalanceOrig'] == 0)
).astype(int)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_amount_spike'] = (`


In [21]:
df['rule_amount_spike'] = (
    df['amount'] > (df['avg_sent'] * 3)
).astype(int)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_velocity'] = (df['txn_per_step'] > 1).astype(int)`


In [22]:
df['rule_velocity'] = (df['txn_per_step'] > 1).astype(int)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `rule_cols = [`


In [23]:
rule_cols = [
    'rule_high_value',
    'rule_high_risk_type',
    'rule_account_emptied',
    'rule_amount_spike',
    'rule_velocity'
]

df['rule_score'] = df[rule_cols].sum(axis=1)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_based_alert'] = (df['rule_score'] >= 2).astype(int)`


In [24]:
df['rule_based_alert'] = (df['rule_score'] >= 2).astype(int)

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_based_alert'].value_counts()`


In [25]:
df['rule_based_alert'].value_counts()

rule_based_alert
0    4583199
1    1779421
Name: count, dtype: int64

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `pd.crosstab(df['rule_based_alert'], df['isFraud'])`


In [26]:
pd.crosstab(df['rule_based_alert'], df['isFraud'])

isFraud,0,1
rule_based_alert,,
0,4583172,27
1,1771235,8186


### Code Breakdown

**What this cell does:** Builds aggregated views to summarize behavior across entities.

**Key command:** `df.groupby('rule_based_alert')['isFraud'].mean()`


In [27]:
df.groupby('rule_based_alert')['isFraud'].mean()

rule_based_alert
0    0.000006
1    0.004600
Name: isFraud, dtype: float64

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `suspicious = df[df['rule_based_alert'] == 1]`


In [28]:
suspicious = df[df['rule_based_alert'] == 1]

suspicious.to_csv('../reports/suspicious_transactions.csv', index=False)
df.to_csv('../data/processed/transactions_rules.csv', index=False)

## Rule-Based Detection Summary

This notebook created an initial AML rule-based detection layer using behavioural and transaction-based indicators.

Rules created:
- High-value transaction
- High-risk transaction type
- Account emptied after transaction
- Amount spike compared to sender average
- Multiple transactions within the same time step

The rule score combines these indicators and flags transactions with two or more suspicious signals for review.

This simulates a first-line AML monitoring system where alerts are generated before machine learning prioritisation.

## Rule Tuning

### Code Breakdown

**What this cell does:** Builds aggregated views to summarize behavior across entities.

**Key command:** `rule_performance = df.groupby('rule_score').agg(`


In [29]:
rule_performance = df.groupby('rule_score').agg(
    total_transactions=('isFraud', 'count'),
    fraud_cases=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean')
).reset_index()

rule_performance

,rule_score,total_transactions,fraud_cases,fraud_rate
0,0,2783781,0,0.000000
1,1,1799418,27,0.000015
2,2,1173036,2889,0.002463
3,3,606376,5297,0.008736
4,4,9,0,0.000000


### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `df['rule_alert_strict'] = (df['rule_score'] >= 3).astype(int)`


In [30]:
df['rule_alert_strict'] = (df['rule_score'] >= 3).astype(int)

df['rule_alert_strict'].value_counts()

rule_alert_strict
0    5756235
1     606385
Name: count, dtype: int64

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `pd.crosstab(df['rule_alert_strict'], df['isFraud'])`


In [31]:
pd.crosstab(df['rule_alert_strict'], df['isFraud'])

isFraud,0,1
rule_alert_strict,,
0,5753319,2916
1,601088,5297


### Code Breakdown

**What this cell does:** Builds aggregated views to summarize behavior across entities.

**Key command:** `df.groupby('rule_alert_strict')['isFraud'].mean()`


In [32]:
df.groupby('rule_alert_strict')['isFraud'].mean()

rule_alert_strict
0    0.000507
1    0.008735
Name: isFraud, dtype: float64

### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `original_alerts = df[df['rule_based_alert'] == 1]`


In [33]:
original_alerts = df[df['rule_based_alert'] == 1]
strict_alerts = df[df['rule_alert_strict'] == 1]

comparison = pd.DataFrame({
    'scenario': ['Original Rules: score >= 2', 'Strict Rules: score >= 3'],
    'alerts_created': [len(original_alerts), len(strict_alerts)],
    'fraud_captured': [original_alerts['isFraud'].sum(), strict_alerts['isFraud'].sum()],
    'alert_fraud_rate': [original_alerts['isFraud'].mean(), strict_alerts['isFraud'].mean()]
})

comparison

,scenario,alerts_created,fraud_captured,alert_fraud_rate
0,Original Rules: score >= 2,1779421,8186,0.004600
1,Strict Rules: score >= 3,606385,5297,0.008735


### Code Breakdown

**What this cell does:** Executes a processing step in the AML workflow.

**Key command:** `comparison.to_csv('../reports/rule_tuning_comparison.csv', index=False)`


In [34]:
comparison.to_csv('../reports/rule_tuning_comparison.csv', index=False)
df.to_csv('../data/processed/transactions_rules.csv', index=False)

## Rule Tuning Summary

The first rule-based alert threshold captured almost all fraud cases but generated a very large number of alerts. This reflects a common AML challenge where detection systems create high false positives.

A stricter rule threshold was tested to reduce investigation workload. The comparison report shows the trade-off between fraud capture and alert volume, which is central to AML monitoring and alert prioritisation.